# 🌍 AI Travel Planner — Jupyter Notebook

This notebook is used to **understand and test** the AI logic before integrating it into the FastAPI backend.

## What we'll do:
1. Install and import dependencies
2. Set up the LLM (OpenRouter via LangChain)
3. Build a travel itinerary prompt
4. Call the LLM and see the output
5. Test with different destinations

## Step 1: Install Dependencies

In [ ]:
# Run this once to install required packages
!pip install openai langchain langchain-openai python-dotenv

## Step 2: Set Up OpenRouter API Key & LLM

> **OpenRouter** is OpenAI-API compatible. You just change `openai_api_base` and `openai_api_key`.
> Get your key at https://openrouter.ai/keys

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# ─────────────────────────────────────────────────────────
# OpenRouter Setup (OpenAI-compatible — no extra SDK needed)
# ─────────────────────────────────────────────────────────
OPENROUTER_API_KEY = "Enter your API Key"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 💡 Model options you can try on OpenRouter:
#   'openai/gpt-3.5-turbo'         ← cheapest OpenAI model
#   'openai/gpt-4o-mini'           ← smarter, still cheap
#   'mistralai/mistral-7b-instruct' ← very cheap
#   'google/gemma-3-27b-it:free'   ← completely FREE!
MODEL = "openai/gpt-3.5-turbo"

# Initialize LLM via OpenRouter
llm = ChatOpenAI(
    model=MODEL,
    temperature=0.7,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": "http://localhost",   # Required by OpenRouter
        "X-Title": "AI Travel Planner",       # Shows in your OpenRouter dashboard
    }
)

print("✅ OpenRouter LLM initialized successfully!")
print(f"   Model : {MODEL}")
print(f"   Key   : {OPENROUTER_API_KEY[:20]}...")

## Step 3: Build the Prompt

In [ ]:
# Travel parameters — change these to experiment!
destination = "Goa"
preferences = "beach, nightlife, budget food"
days = 3
budget = "budget (under ₹2000/day per person)"
travelers = 2

prompt = f"""
You are an expert AI travel planner with deep knowledge of Indian and international destinations.

Create a detailed {days}-day travel itinerary for the following trip:

📍 Destination: {destination}
🎯 Preferences: {preferences}
👥 Travelers: {travelers} person(s)
💰 Budget Level: {budget}

Please provide a comprehensive travel plan including:
1. Day-by-day itinerary (Morning, Afternoon, Evening for each day)
2. 3 hotel suggestions with approximate prices in INR
3. Local food & restaurant recommendations
4. Full cost breakdown in INR for {travelers} travelers, {days} days
5. 3 pro tips specific to this destination

Use emojis to make it engaging and be specific with names of places!
"""

print("📋 Prompt built! Length:", len(prompt), "characters")
print("\n" + "="*60)
print(prompt)
print("="*60)

## Step 4: Call the LLM via OpenRouter

In [ ]:
print(f"🚀 Sending request to OpenRouter ({MODEL})...\n")

messages = [HumanMessage(content=prompt)]
response = llm.invoke(messages)

itinerary = response.content

print("✅ AI Response Generated!")
print("\n" + "="*60)
print(itinerary)
print("="*60)

## Step 5: Test with Different Destinations

In [ ]:
def generate_itinerary(destination, preferences, days=3, budget="budget", travelers=2):
    """Reusable function to generate travel itinerary via OpenRouter."""
    prompt = f"""
    Create a concise {days}-day travel itinerary for:
    - Destination: {destination}
    - Preferences: {preferences}
    - Travelers: {travelers}
    - Budget: {budget}
    
    Include: day-wise plan, 2 hotel options, key food spots, and total estimated cost in INR.
    """
    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)
    return response.content

# Test 1: Manali
print("🏔️ Generating Manali itinerary...\n")
result = generate_itinerary("Manali", "mountains, snow, adventure", days=4, budget="medium", travelers=2)
print(result)

In [ ]:
# Test 2: Kerala
print("🌴 Generating Kerala itinerary...\n")
result = generate_itinerary("Kerala", "backwaters, nature, ayurveda", days=5, budget="luxury", travelers=2)
print(result)

## 💡 Switch to a FREE Model

To use a **completely free** model on OpenRouter, change the `MODEL` variable in Step 2:

In [ ]:
# Switch to a FREE model (no credits needed)
free_llm = ChatOpenAI(
    model="google/gemma-3-27b-it:free",
    temperature=0.7,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "AI Travel Planner",
    }
)

print("🆓 Testing with free model: google/gemma-3-27b-it...\n")
messages = [HumanMessage(content="Give a quick 2-day itinerary for Jaipur, India. Budget traveler.")]
res = free_llm.invoke(messages)
print(res.content)

## 🎯 Next Steps

Now that you understand how the AI works:
1. ✅ This logic is in `app/services/llm_service.py` (already uses OpenRouter)
2. ✅ The prompt is in `app/utils/prompt_builder.py`
3. ✅ The FastAPI routes in `app/routes/planner.py` expose it as an API
4. ✅ The HTML frontend calls the API

**Run the backend:**
```bash
uvicorn app.main:app --reload
```